# sample&split

从数据集中进行随机采样和train-eval-test采样

In [1]:
import json, random, os, math

input_file_path = "/home/jiazixiao.jzx/TSRL4jailbreak/data/dataset/processed/deduped_all.jsonl"

# ===== 你只需要改这里 =====
seed = 42

# 选定数据量（None 表示用全量；或填一个整数比如 20000）
sample_size = 1
# 输出目录名（会创建在 base_out_root 下）
out_dir_name = f"s1_{sample_size}k"
sample_size*=1000

# 划分比例
train_ratio, eval_ratio, test_ratio = 0.6, 0.2, 0.2
assert abs(train_ratio + eval_ratio + test_ratio - 1.0) < 1e-9

base_out_root = "/home/jiazixiao.jzx/TSRL4jailbreak/data/dataset/processed"
# ==========================

out_dir = os.path.join(base_out_root, out_dir_name)
train_out = os.path.join(out_dir, "train.jsonl")
eval_out  = os.path.join(out_dir, "eval.jsonl")
test_out  = os.path.join(out_dir, "test.jsonl")

# ===== 读取 =====
data = []
with open(input_file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

n_total = len(data)
print("Total available:", n_total)

# ===== 抽样 =====
random.seed(seed)
if sample_size is None or sample_size >= n_total:
    sampled = data[:]  # 全量
    print("Using full dataset")
else:
    sampled = random.sample(data, sample_size)
    print(f"Sampled: {len(sampled)}")

# ===== 打乱 + 切分 =====
random.shuffle(sampled)
n = len(sampled)

n_train = int(n * train_ratio)
n_eval  = int(n * eval_ratio)
n_test  = n - n_train - n_eval

train_data = sampled[:n_train]
eval_data  = sampled[n_train:n_train + n_eval]
test_data  = sampled[n_train + n_eval:]

print(f"Split sizes -> train={len(train_data)}, eval={len(eval_data)}, test={len(test_data)}")

# ===== 补 id（可选但推荐）=====
def ensure_ids(items, split):
    for i, it in enumerate(items):
        if "id" not in it or it["id"] is None or str(it["id"]).strip() == "":
            it["id"] = f"{out_dir_name}:{split}:{i}"

ensure_ids(train_data, "train")
ensure_ids(eval_data, "eval")
ensure_ids(test_data, "test")

# ===== 写出 =====
os.makedirs(out_dir, exist_ok=True)

def write_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")

write_jsonl(train_out, train_data)
write_jsonl(eval_out,  eval_data)
write_jsonl(test_out,  test_data)

print("Saved to:")
print(" ", train_out)
print(" ", eval_out)
print(" ", test_out)


Total available: 36281
Sampled: 1000
Split sizes -> train=600, eval=200, test=200
Saved to:
  /home/jiazixiao.jzx/TSRL4jailbreak/data/dataset/processed/s1_1k/train.jsonl
  /home/jiazixiao.jzx/TSRL4jailbreak/data/dataset/processed/s1_1k/eval.jsonl
  /home/jiazixiao.jzx/TSRL4jailbreak/data/dataset/processed/s1_1k/test.jsonl
